# 05-7. NumPy 배열 — 풀이 검증

## Goal

결측값·조건 마스크·벡터 집계를 검증한다.

> 학습자용 TODO를 먼저 완성한 뒤 참고한다.


## Setup

fixture와 실행 환경을 확인한다.


In [ ]:
from pathlib import Path
import sys


def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "requirements.txt").is_file():
            return candidate
    raise FileNotFoundError("requirements.txt가 있는 저장소 루트에서 JupyterLab을 실행하세요.")


ROOT = find_project_root()
FIXTURE_DIR = ROOT / "fixtures" / "05-text-processing"

assert sys.version_info >= (3, 10)
assert FIXTURE_DIR.is_dir()

print("Python:", sys.version.split()[0])
print("실습 데이터:", FIXTURE_DIR)


import numpy as np
fixture_path = FIXTURE_DIR / "measurements.csv"
values = np.genfromtxt(fixture_path, delimiter=",", skip_header=1, usecols=1, dtype=float)


## Steps

참고 구현을 실행한다.


In [ ]:
valid_mask = np.isfinite(values)
valid_values = values[valid_mask]
valid_mean = float(valid_values.mean())
high_mask = valid_mask & (values >= 80)

total_requests = np.array([10, 40, 5, 0], dtype=np.int64)
not_found_404 = np.array([9, 9, 0, 0], dtype=np.int64)
unique_404_paths = np.array([9, 1, 0, 0], dtype=np.int64)
sensitive_requests = np.array([0, 0, 1, 0], dtype=np.int64)
unique_sensitive_paths = np.array([0, 0, 1, 0], dtype=np.int64)
sensitive_2xx = np.array([0, 0, 1, 0], dtype=np.int64)

not_found_rate = np.divide(
    not_found_404,
    total_requests,
    out=np.zeros(total_requests.shape, dtype=float),
    where=total_requests > 0,
)
scan_signal = (
    (not_found_404 >= 8)
    & (not_found_rate >= 0.70)
    & (unique_404_paths >= 6)
)
sensitive_signal = (
    (sensitive_2xx >= 1)
    | ((sensitive_requests >= 3) & (unique_sensitive_paths >= 2))
)
candidate = scan_signal | sensitive_signal
reason = np.select(
    [scan_signal & sensitive_signal, sensitive_signal, scan_signal],
    ["scan+sensitive", "sensitive", "scan"],
    default="-",
)

print("values:", values)
print("valid_mean:", valid_mean)
print("high:", values[high_mask])
print("not_found_rate:", not_found_rate)
print("reason:", reason)


## Checks

경계값과 fixture 결과를 대조한다.


In [ ]:
assert valid_mask.dtype == np.bool_
assert int(valid_mask.sum()) == 4
assert np.isclose(valid_mean, 30.5)
assert values[high_mask].tolist() == [85.0]
assert np.isnan(values[2]) and np.isnan(values[4])
assert np.allclose(not_found_rate, [0.9, 0.225, 0.0, 0.0])
assert candidate.tolist() == [True, False, True, False]
assert reason.tolist() == ["scan", "-", "sensitive", "-"]
print("검증 통과")


## Next Steps

다중 조건이 있을 때 간단한 bool 배열을 먼저 이름 붙여 만들고 결합한다.
